# 01 · Run one complete model day

Choose one `MODEL_ID`, review the derived status, then explicitly enable the
expensive stage. Rerunning all cells resumes only compatible unfinished work.
Search rungs, final checkpoints, evaluation, profiling, and reports are
discovered automatically from persistent storage.


In [1]:
MODEL_ID = "rtdetrv2_l"

RUN_MODE = "auto"
RUN_LR_RANGE_TEST = True
RUN_BOUNDARY_EXTENSION = False

START_EXPENSIVE_STAGE = True
ALLOW_OVER_BUDGET_RUN = False
DATA_ACCESS_MODE = "local_cache"


In [2]:
import importlib.util
import json
import os
import subprocess
import sys
from pathlib import Path

try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
REPO_PATH = Path("/content/aerial-object-detection-benchmark") if IN_COLAB else Path.cwd()
if IN_COLAB:
    if not (REPO_PATH / ".git").is_dir():
        subprocess.run(
            ["git", "clone", "--branch", "main", "https://github.com/Harryphan72007/aerial-object-detection-benchmark.git", str(REPO_PATH)],
            check=True,
        )
    elif subprocess.check_output(["git", "-C", str(REPO_PATH), "status", "--porcelain"], text=True).strip():
        raise RuntimeError("Repository has local changes; refusing to update it.")
    else:
        subprocess.run(["git", "-C", str(REPO_PATH), "pull", "--ff-only", "origin", "main"], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r", str(REPO_PATH / "requirements-dataset-colab.txt")],
        check=True,
    )
sys.path.insert(0, str(REPO_PATH))
DRIVE_ROOT = (
    Path("/content/drive/MyDrive/visdrone_architecture_benchmark")
    if IN_COLAB
    else Path(os.environ.get("VISDRONE_DRIVE_ROOT", REPO_PATH / "local_artifacts"))
)
SMOKE_TEST = os.environ.get("SMOKE_TEST", "").lower() in {"1", "true", "yes"}
RESOLVED_DATA_ACCESS_MODE = DATA_ACCESS_MODE if IN_COLAB else "drive_direct"
LOCAL_CACHE_ROOT = Path("/content/visdrone_cache")


Mounted at /content/drive


In [3]:
from src.data.contract import verify_complete_data_contract
from src.data.local_cache import resolve_data_access
from src.paths import ProjectPaths
from src.workflows.model_day import inspect_model_day

paths = ProjectPaths.from_value(DRIVE_ROOT)
if SMOKE_TEST:
    from src.data.download import VISDRONE_ARCHIVES
    for spec in VISDRONE_ARCHIVES.values():
        spec["minimum_bytes"] = 1
max_images = 12 if SMOKE_TEST else None

def require_verified_data(report):
    if report.verified:
        return
    print("DATA CONTRACT VERIFIED: NO")
    print("Dataset setup is incomplete or was interrupted.")
    if any("staging directory" in error for error in report.errors):
        print("A stale extraction staging directory was detected; notebook 00 will recover or rebuild it from the verified Drive ZIP.")
    print("Run every cell in 00_prepare_visdrone.ipynb, wait for DATA CONTRACT VERIFIED: YES, then rerun notebook 01.")
    print("https://colab.research.google.com/github/Harryphan72007/aerial-object-detection-benchmark/blob/main/notebooks/00_prepare_visdrone.ipynb")
    raise RuntimeError("Notebook 01 stopped safely before caching or training because dataset preparation is incomplete.")

if RESOLVED_DATA_ACCESS_MODE == "local_cache":
    persistent_contract = verify_complete_data_contract(
        paths,
        repo_root=REPO_PATH,
        max_images_per_split=max_images,
    )
    require_verified_data(persistent_contract)
data_access = resolve_data_access(
    paths,
    RESOLVED_DATA_ACCESS_MODE,
    cache_root=LOCAL_CACHE_ROOT,
)
contract = verify_complete_data_contract(
    paths,
    repo_root=REPO_PATH,
    local_cache=data_access if data_access.mode == "local_cache" else None,
    max_images_per_split=max_images,
)
require_verified_data(contract)
print("DATA CONTRACT VERIFIED: YES")
print(json.dumps(contract.to_dict(), indent=2, default=str))

state = inspect_model_day(
    DRIVE_ROOT,
    MODEL_ID,
    REPO_PATH,
    verify_data=False,
)
state["data_contract"] = contract.to_dict()
print(json.dumps(state, indent=2, default=str))
if state["stage"] == "DATA":
    raise RuntimeError(
        "Dataset setup is incomplete. Run 00_prepare_visdrone.ipynb, then rerun this notebook."
    )


{
  "local_cache": "VERIFIED",
  "root": "/content/visdrone_cache",
  "size_gib": 1.807,
  "copy_seconds": 290.086,
  "copied_files": 7032
}
DATA CONTRACT VERIFIED: YES
{
  "verified": true,
  "checks": {
    "drive_root_mounted_and_writable": true,
    "train_archive_valid": true,
    "train_extraction_manifest_matches": true,
    "2class_train_conversion_current": true,
    "val_archive_valid": true,
    "val_extraction_manifest_matches": true,
    "2class_val_conversion_current": true,
    "lr_search_manifests_current": true,
    "lr_search_numeric_ids_disjoint": true,
    "lr_search_train_subset_official_train": true,
    "lr_search_validation_subset_official_train": true,
    "lr_official_train_validation_numeric_ids_disjoint": true,
    "lr_search_train_validation_filenames_disjoint": true,
    "lr_search_train_filenames_subset_official_train": true,
    "lr_search_validation_filenames_subset_official_train": true,
    "lr_official_train_validation_filenames_disjoint": true,
    

In [4]:
from src.workflows.model_day import ModelDayOptions, run_model_day
result = run_model_day(
    REPO_PATH,
    DRIVE_ROOT,
    ModelDayOptions(
        model_id=MODEL_ID,
        run_mode=RUN_MODE,
        run_lr_range_test=RUN_LR_RANGE_TEST,
        run_boundary_extension=RUN_BOUNDARY_EXTENSION,
        start_expensive_stage=START_EXPENSIVE_STAGE,
        allow_over_budget_run=ALLOW_OVER_BUDGET_RUN,
        smoke_test=SMOKE_TEST,
        data_access_mode=RESOLVED_DATA_ACCESS_MODE,
        local_cache_root=str(LOCAL_CACHE_ROOT),
    ),
    data_access=data_access,
    verified_data_contract=contract.to_dict(),
)
print(json.dumps(result, indent=2, default=str))


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

{
  "model_id": "rtdetrv2_l",
  "environment_preflight": "BLOCKED",
  "lr_search_status": "NOT_STARTED",
  "selected_lr": null,
  "selected_config": null,
  "final_training_status": "NOT_STARTED",
  "final_run_id": null,
  "final_run_dir": null,
  "best_checkpoint": null,
  "evaluation_status": "NOT_STARTED",
  "evaluation_files": [],
  "report_status": "NOT_STARTED",
  "report_path": null,
  "bundle_status": "NOT_STARTED",
  "bundle_id": null,
  "bundle_path": null,
  "search_completed_rungs": [],
  "stage": "ENVIRONMENT",
  "drive_root": "/content/drive/MyDrive/visdrone_architecture_benchmark",
  "dataset_train": "/content/drive/MyDrive/visdrone_architecture_benchmark/datasets/VisDrone2019-DET/processed/coco_2class/annotations/instances_train.json",
  "dataset_validation": "/content/drive/MyDrive/visdrone_architecture_benchmark/datasets/VisDrone2019-DET/processed/coco_2class/annotations/instances_val.json",
  "train_images": "/content/drive/MyDrive/visdrone_architecture_benchmark/dat

In [5]:
if result["stage"] == "COMPLETE":
    evaluation = json.loads(Path(result["evaluation_paths"][0]).read_text())
    print("MODEL DAY COMPLETE")
    print()
    print(f"Model: {MODEL_ID}")
    print(f"Selected LR: {result['selected_lr']}")
    print(f"Final run ID: {result['final_run_id']}")
    print(f"Best checkpoint: {result['checkpoint_path']}")
    print(f"mAP50-95: {evaluation.get('mAP')}")
    print(f"APtiny: {evaluation.get('APtiny')}")
    print(f"Report: {result['report_path']}")
    print(f"Recommended bundle: {result['recommended_bundle']}")
    print("Next notebook: 02_publish_results.ipynb")
else:
    print(result.get("message", f"Next stage: {result['stage']}"))


Core packages changed. Restart the Colab session once, then rerun this notebook from the top.
